# Hybrid Training - ResNet18 + CBAM

Treino do modelo híbrido ResNet18 + CBAM Attention no dataset PCam.

**Tempo esperado:** ~45-60 minutos (GPU T4)

---

## 1. Setup

In [ ]:
# Verificar GPU
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Navegar para o projeto
%cd /content/drive/MyDrive/histopathology-cancer-cell-detection

## 2. Imports

In [ ]:
import sys
import json
from pathlib import Path

# Adicionar src ao path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import torch
from src.data.dataset import get_dataloaders
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.hybrid import create_hybrid_model
from src.training.trainer import Trainer
from src.utils.logger import setup_logger

print(" Imports completos")

## 3. Carregar Configuração

In [ ]:
# Carregar config
with open('configs/hybrid.json', 'r') as f:
    config = json.load(f)

# Forçar CUDA
config['device'] = 'cuda'

print(" Configuração:")
print(f"   Modelo: {config['model']['architecture']}")
print(f"   CBAM reduction: {config['model']['cbam']['reduction_ratio']}")
print(f"   CBAM kernel: {config['model']['cbam']['kernel_size']}")
print(f"   Épocas: {config['training']['epochs']}")
print(f"   Batch size: {config['data']['batch_size']}")
print(f"   Learning rate: {config['training']['learning_rate']}")

## 4. Criar DataLoaders

In [ ]:
# Criar transforms
train_transform = get_train_transforms(config)
val_transform = get_val_transforms(config)

# Criar dataloaders
train_loader, val_loader = get_dataloaders(
    config,
    train_transform,
    val_transform
)

print(f"\n DataLoaders criados")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

## 5. Criar Modelo Híbrido

In [ ]:
# Criar modelo híbrido (ResNet18 + CBAM)
model = create_hybrid_model(config)

# Contar parâmetros
params = model.count_parameters()
print(f"\n Modelo híbrido criado")
print(f"   Total parameters: {params['total']:,}")
print(f"   Trainable: {params['trainable']:,}")
print(f"   CBAM parameters: {params['cbam']:,}")

## 6. Setup Logger

In [ ]:
# Setup logger
logger = setup_logger('hybrid_resnet18_cbam', log_dir='logs')
print("\n Logger configurado")

## 7. Criar Trainer

In [ ]:
# Criar trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    save_dir=config['checkpoints']['save_dir']
)

print("\n Trainer criado")
print(f"   Checkpoints: {config['checkpoints']['save_dir']}")

## 8.  TREINAR


In [ ]:
# TREINAR!
num_epochs = config['training']['epochs']

print("\n" + "="*70)
print(f" INICIANDO TREINO HÍBRIDO - {num_epochs} ÉPOCAS")
print("= *70 + "\n")

trainer.train(num_epochs)

print("\n" + "="*70)
print(" TREINO HÍBRIDO COMPLETO!")
print("="*70)

## 9. Resultados

In [ ]:
import matplotlib.pyplot as plt

# Carregar histórico
history = trainer.history

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history['val_accuracy'], label='Accuracy', color='green', linewidth=2)
axes[1].plot(history['val_f1'], label='F1-Score', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(history['learning_rate'], color='red', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hybrid_training.png', dpi=300, bbox_inches='tight')
plt.show()

# Estatísticas
print("\n RESULTADOS FINAIS:")
print("="*50)
print(f"Best Epoch:      {trainer.best_epoch}")
print(f"Best Accuracy:   {trainer.best_metric:.4f}")
print(f"Final Val Loss:  {history['val_loss'][-1]:.4f}")
print(f"Final Val AUC:   {history['val_auc_roc'][-1]:.4f}")
print("="*50)

## 10. Comparar com Baseline

In [ ]:
# Carregar baseline history
try:
    baseline_hist_path = config['checkpoints']['save_dir'].replace('hybrid', 'baseline') + '/history.json'
    with open(baseline_hist_path, 'r') as f:
        baseline_history = json.load(f)
    
    # Comparação
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Accuracy comparison
    axes[0].plot(baseline_history['val_accuracy'], label='Baseline', linewidth=2, color='blue')
    axes[0].plot(history['val_accuracy'], label='Hybrid (CBAM)', linewidth=2, color='red')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Validation Accuracy')
    axes[0].set_title('Baseline vs Hybrid - Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # F1 comparison
    axes[1].plot(baseline_history['val_f1'], label='Baseline', linewidth=2, color='blue')
    axes[1].plot(history['val_f1'], label='Hybrid (CBAM)', linewidth=2, color='red')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation F1-Score')
    axes[1].set_title('Baseline vs Hybrid - F1')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('baseline_vs_hybrid.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Estatísticas comparativas
    baseline_best = max(baseline_history['val_accuracy'])
    hybrid_best = trainer.best_metric
    improvement = hybrid_best - baseline_best
    
    print("\n COMPARAÇÃO BASELINE vs HÍBRIDO:")
    print("="*50)
    print(f"Baseline Best Acc:  {baseline_best:.4f}")
    print(f"Hybrid Best Acc:    {hybrid_best:.4f}")
    print(f"Ganho (Δ):          {improvement:.4f} (+{improvement*100:.2f}%)")
    print("="*50)
    
except FileNotFoundError:
    print("  Baseline history não encontrado")
    print("   Treina o baseline primeiro!")

## 11. Salvar Resultados Finais

In [ ]:
# Salvar summary
summary = {
    'model': 'ResNet18 + CBAM (Hybrid)',
    'best_epoch': trainer.best_epoch,
    'best_accuracy': float(trainer.best_metric),
    'total_epochs': len(history['train_loss']),
    'cbam_params': params['cbam'],
    'final_metrics': {
        'accuracy': float(history['val_accuracy'][-1]),
        'precision': float(history['val_precision'][-1]),
        'recall': float(history['val_recall'][-1]),
        'f1': float(history['val_f1'][-1]),
        'auc_roc': float(history['val_auc_roc'][-1])
    }
}

import json
with open('hybrid_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(" Summary salvo em: hybrid_summary.json")
print(" Gráfico salvo em: hybrid_training.png")
print(" Comparação salva em: baseline_vs_hybrid.png")
print(f" Checkpoints em: {config['checkpoints']['save_dir']}")